# 🏥 Multimodal Medical Foundation Model
## Image + Text Branch using BiomedCLIP and BioClinicalBERT

---

### Project Overview

This notebook implements a **multimodal chest X-ray disease classification system** that jointly learns from:

- **Chest X-ray images** — encoded via [BiomedCLIP](https://huggingface.co/microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224) (ViT-B/16)
- **Radiology reports** — encoded via [BioClinicalBERT](https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT)

The model performs **multi-label classification** for 6 pathologies:

| Index | Disease |
|-------|---------|
| 0 | No Finding |
| 1 | Cardiomegaly |
| 2 | Edema |
| 3 | Pleural Effusion |
| 4 | Pneumonia |
| 5 | Pneumothorax |

### Key Features

- ✅ BiomedCLIP image encoder with partial fine-tuning
- ✅ BioClinicalBERT text encoder with partial fine-tuning
- ✅ Late fusion with deep classification head
- ✅ Focal loss with class-weight balancing
- ✅ Mixed precision (AMP), EMA, early stopping
- ✅ GradCAM explainability for ViT
- ✅ Integrated Gradients for text attribution
- ✅ Threshold optimization and TTA
- ✅ Full artifact saving pipeline

---

## 1. Environment Setup & Installation

In [ ]:
%%capture
!pip install open_clip_torch captum --quiet
!pip install transformers --quiet

In [ ]:
# ============================================================
# 1.1 Imports
# ============================================================

import os
import gc
import json
import copy
import math
import time
import random
import pickle
import warnings
import re
from pathlib import Path
from collections import OrderedDict
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T

from transformers import AutoTokenizer, AutoModel
import open_clip

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    hamming_loss,
)

from captum.attr import IntegratedGradients, LayerIntegratedGradients

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("✅ All imports successful.")

## 2. Configuration

In [ ]:
# ============================================================
# 2.1 Configuration Dictionary
# ============================================================

CONFIG = {
    # ── Paths ──────────────────────────────────────────────────
       "image_root": "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/official_data_iccv_final/files",
    "train_csv":  "/kaggle/input/datasets/vedantkulkarni14/labeled-mimic-cxr/labeled_mimic_cxr/mimic_final_labeled_train.csv",
    "val_csv":    "/kaggle/input/datasets/vedantkulkarni14/labeled-mimic-cxr/labeled_mimic_cxr/mimic_final_labeled_validate.csv",
    "output_dir": "/kaggle/working/artifacts",

    # ── Disease labels ─────────────────────────────────────────
    "disease_labels": [
        "No Finding",
        "Cardiomegaly",
        "Edema",
        "Pleural Effusion",
        "Pneumonia",
        "Pneumothorax",
    ],
    "num_classes": 6,

    # ── Model identifiers ──────────────────────────────────────
    "biomed_clip_model": "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    "text_model_name": "emilyalsentzer/Bio_ClinicalBERT",

    # ── Dimensions ─────────────────────────────────────────────
    "image_embed_dim": 512,
    "text_embed_dim": 768,
    "fused_dim": 1280,  # 512 + 768

    # ── Text ───────────────────────────────────────────────────
    "max_text_len": 256,

    # ── Image ──────────────────────────────────────────────────
    "image_size": 224,

    # ── Training ───────────────────────────────────────────────
    "epochs": 20,
    "batch_size": 32,
    "num_workers": 4,
    "patience": 6,
    "max_grad_norm": 1.0,
    "ema_decay": 0.999,
    "mixed_precision": True,

    # ── Learning rates ─────────────────────────────────────────
    "lr_image_encoder": 5e-6,
    "lr_text_encoder": 2e-5,
    "lr_fusion_head": 1e-4,
    "weight_decay": 1e-4,

    # ── Loss ───────────────────────────────────────────────────
    "focal_gamma": 2.0,
    "weight_clip_min": 0.5,
    "weight_clip_max": 20.0,

    # ── Freeze strategy ────────────────────────────────────────
    "biomed_clip_freeze_blocks": 6,   # Freeze first 6 ViT blocks
    "bert_freeze_layers": 10,         # Freeze first 10 BERT layers

    # ── Threshold search ───────────────────────────────────────
    "threshold_start": 0.10,
    "threshold_end": 0.90,
    "threshold_steps": 81,

    # ── Reproducibility ────────────────────────────────────────
    "seed": 42,
}

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("✅ Configuration loaded.")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

In [ ]:
# ============================================================
# 2.2 Seed Everything for Reproducibility
# ============================================================

def seed_everything(seed: int = 42):
    """Set random seed for reproducibility across all libraries."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✅ Seed set to {seed}")

seed_everything(CONFIG["seed"])

In [ ]:
# ============================================================
# 2.3 Device Detection
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()

print(f"🖥️  Device: {DEVICE}")
print(f"🖥️  Number of GPUs: {NUM_GPUS}")
if torch.cuda.is_available():
    for i in range(NUM_GPUS):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"   Memory: {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")

---
## 3. Data Preparation

We load the MIMIC-CXR labeled CSVs, clean reports, verify image paths, and filter to keep only complete samples.

In [ ]:
# ============================================================
# 3.1 Load CSVs
# ============================================================

df_train_raw = pd.read_csv(CONFIG["train_csv"])
df_val_raw = pd.read_csv(CONFIG["val_csv"])

print(f"📊 Raw train samples: {len(df_train_raw)}")
print(f"📊 Raw validation samples: {len(df_val_raw)}")
print(f"\n📋 Train columns: {list(df_train_raw.columns)}")
print(f"\n🔍 Train sample (first row):")
df_train_raw.head(2)

In [ ]:
# ============================================================
# 3.2 Report Cleaning
# ============================================================

def clean_report(text: str) -> str:
    """Clean a radiology report text.
    
    Steps:
        1. Convert to lowercase
        2. Remove extra whitespace
        3. Strip leading/trailing spaces
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)  # Remove duplicate/extra spaces
    text = text.strip()
    return text


def prepare_dataframe(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """Clean, filter, and validate a dataframe.
    
    Steps:
        1. Clean report text
        2. Remove missing/empty/short reports
        3. Verify image existence and auto-detect paths
        4. Keep only samples with both image and report
    """
    initial_count = len(df)
    print(f"\n{'='*60}")
    print(f"🔧 Preparing {split_name} split ({initial_count} samples)")
    print(f"{'='*60}")

    # --- Step 1: Clean reports ---
    df = df.copy()
    df["text"] = df["text"].apply(clean_report)
    print(f"  ✅ Reports cleaned.")

    # --- Step 2: Remove missing / empty / short reports ---
    mask_missing = df["text"].isna() | (df["text"] == "")
    n_missing = mask_missing.sum()
    df = df[~mask_missing].copy()
    print(f"  🗑️  Removed {n_missing} missing/empty reports.")

    mask_short = df["text"].str.len() < 10
    n_short = mask_short.sum()
    df = df[~mask_short].copy()
    print(f"  🗑️  Removed {n_short} reports with < 10 characters.")

    # --- Step 3: Auto-detect image paths and verify existence ---
    image_root = Path(CONFIG["image_root"])

    def find_image_path(row):
        """Automatically detect the full image path."""
        img_name = str(row["image"])
        # Try direct path first
        direct_path = image_root / img_name
        if direct_path.exists():
            return str(direct_path)
        # Try with subject/study structure
        subject_id = str(row.get("subject_id", ""))
        study_id = str(row.get("study_id", ""))
        structured_path = image_root / f"p{subject_id[:2]}" / f"p{subject_id}" / f"s{study_id}" / img_name
        if structured_path.exists():
            return str(structured_path)
        # Try recursive search in image_root (limited)
        for candidate in image_root.rglob(img_name):
            return str(candidate)
        return None

    print(f"  🔍 Verifying image paths...")
    df["image_path"] = df.apply(find_image_path, axis=1)
    
    n_no_image = df["image_path"].isna().sum()
    df = df[df["image_path"].notna()].copy()
    print(f"  🗑️  Removed {n_no_image} samples without valid image path.")

    # --- Step 4: Final stats ---
    final_count = len(df)
    dropped = initial_count - final_count
    print(f"\n  📊 {split_name} Summary:")
    print(f"     Initial:  {initial_count}")
    print(f"     Final:    {final_count}")
    print(f"     Dropped:  {dropped} ({100*dropped/max(initial_count,1):.1f}%)")

    # Reset index
    df = df.reset_index(drop=True)
    return df


df_train = prepare_dataframe(df_train_raw, "Train")
df_val = prepare_dataframe(df_val_raw, "Validation")

print(f"\n{'='*60}")
print(f"📊 FINAL DATASET SUMMARY")
print(f"{'='*60}")
print(f"  Total samples:       {len(df_train) + len(df_val)}")
print(f"  Train samples:       {len(df_train)}")
print(f"  Validation samples:  {len(df_val)}")
total_dropped = (len(df_train_raw) + len(df_val_raw)) - (len(df_train) + len(df_val))
print(f"  Dropped samples:     {total_dropped}")

In [ ]:
# ============================================================
# 3.3 Label Distribution Analysis
# ============================================================

DISEASE_LABELS = CONFIG["disease_labels"]

print("\n📊 Training Set Label Distribution:")
print("-" * 45)
for label in DISEASE_LABELS:
    count = df_train[label].sum()
    pct = 100 * count / len(df_train)
    print(f"  {label:<20s}: {count:>6d} ({pct:>5.1f}%)")

print("\n📊 Validation Set Label Distribution:")
print("-" * 45)
for label in DISEASE_LABELS:
    count = df_val[label].sum()
    pct = 100 * count / len(df_val)
    print(f"  {label:<20s}: {count:>6d} ({pct:>5.1f}%)")

---
## 4. Tokenizer & Image Preprocessing

We set up the BioClinicalBERT tokenizer and the BiomedCLIP image preprocessing pipeline.

In [ ]:
# ============================================================
# 4.1 Text Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(CONFIG["text_model_name"])
print(f"✅ Tokenizer loaded: {CONFIG['text_model_name']}")
print(f"   Vocab size: {tokenizer.vocab_size}")
print(f"   Max length: {CONFIG['max_text_len']}")

In [ ]:
# ============================================================
# 4.2 BiomedCLIP Preprocessing & Normalization
# ============================================================

# Load BiomedCLIP to extract its preprocessing pipeline
_clip_model, _clip_preprocess = open_clip.create_model_from_pretrained(
    CONFIG["biomed_clip_model"]
)

# Extract normalization parameters from the BiomedCLIP preprocessing
# BiomedCLIP uses specific mean/std for its ViT
BIOMED_CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
BIOMED_CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

# Try to extract from the preprocess transforms if available
for t in _clip_preprocess.transforms:
    if isinstance(t, T.Normalize):
        BIOMED_CLIP_MEAN = tuple(t.mean)
        BIOMED_CLIP_STD = tuple(t.std)
        break

print(f"✅ BiomedCLIP normalization:")
print(f"   Mean: {BIOMED_CLIP_MEAN}")
print(f"   Std:  {BIOMED_CLIP_STD}")

# Free the temporary model
del _clip_model, _clip_preprocess
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 4.3 Image Transform Pipelines
# ============================================================

def get_train_transforms(image_size: int = 224) -> T.Compose:
    """Training image augmentation pipeline.
    
    Augmentations:
        - Resize to target size
        - Random horizontal flip (p=0.5)
        - Random affine (rotation ±5°, translation 2%, scale 95%-105%)
        - Color jitter (brightness=0.1, contrast=0.1)
        - BiomedCLIP normalization
    """
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomAffine(
            degrees=5,
            translate=(0.02, 0.02),
            scale=(0.95, 1.05),
        ),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=BIOMED_CLIP_MEAN, std=BIOMED_CLIP_STD),
    ])


def get_val_transforms(image_size: int = 224) -> T.Compose:
    """Validation image pipeline (no augmentation)."""
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=BIOMED_CLIP_MEAN, std=BIOMED_CLIP_STD),
    ])


train_transforms = get_train_transforms(CONFIG["image_size"])
val_transforms = get_val_transforms(CONFIG["image_size"])

print("✅ Image transforms created.")
print(f"\n🔧 Train transforms:\n{train_transforms}")
print(f"\n🔧 Val transforms:\n{val_transforms}")

---
## 5. Dataset & DataLoaders

In [ ]:
# ============================================================
# 5.1 MultimodalDataset
# ============================================================

class MultimodalDataset(Dataset):
    """Multimodal dataset for chest X-ray images and radiology reports.
    
    Returns a dictionary containing:
        - image: preprocessed chest X-ray tensor [3, 224, 224]
        - input_ids: tokenized report input IDs [max_text_len]
        - attention_mask: attention mask for report [max_text_len]
        - labels: multi-hot disease labels [num_classes]
        - study_id: study identifier (int)
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        tokenizer,
        transforms: T.Compose,
        disease_labels: List[str],
        max_text_len: int = 256,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transforms = transforms
        self.disease_labels = disease_labels
        self.max_text_len = max_text_len

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        # ── Image ──────────────────────────────────────────────
        image_path = row["image_path"]
        image = Image.open(image_path).convert("RGB")
        image = self.transforms(image)

        # ── Text ───────────────────────────────────────────────
        text = str(row["text"])
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)        # [max_text_len]
        attention_mask = encoding["attention_mask"].squeeze(0)  # [max_text_len]

        # ── Labels ─────────────────────────────────────────────
        labels = torch.tensor(
            [float(row[label]) for label in self.disease_labels],
            dtype=torch.float32,
        )

        # ── Study ID ───────────────────────────────────────────
        study_id = int(row["study_id"])

        return {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "study_id": study_id,
        }


print("✅ MultimodalDataset class defined.")

In [ ]:
# ============================================================
# 5.2 Create Datasets and DataLoaders
# ============================================================

train_dataset = MultimodalDataset(
    dataframe=df_train,
    tokenizer=tokenizer,
    transforms=train_transforms,
    disease_labels=DISEASE_LABELS,
    max_text_len=CONFIG["max_text_len"],
)

val_dataset = MultimodalDataset(
    dataframe=df_val,
    tokenizer=tokenizer,
    transforms=val_transforms,
    disease_labels=DISEASE_LABELS,
    max_text_len=CONFIG["max_text_len"],
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=False,
)

print(f"✅ Datasets created.")
print(f"   Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"   Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

# Verify a single sample
sample = train_dataset[0]
print(f"\n🔍 Sample verification:")
print(f"   Image shape:          {sample['image'].shape}")
print(f"   Input IDs shape:      {sample['input_ids'].shape}")
print(f"   Attention mask shape: {sample['attention_mask'].shape}")
print(f"   Labels shape:         {sample['labels'].shape}")
print(f"   Labels:               {sample['labels']}")
print(f"   Study ID:             {sample['study_id']}")

---
## 6. Model Architecture

The model consists of three components:

1. **Image Encoder**: BiomedCLIP ViT-B/16 (512-d embeddings)
2. **Text Encoder**: BioClinicalBERT (768-d CLS embeddings)
3. **Fusion Head**: Deep MLP classifier on concatenated embeddings (1280 → 6)

In [ ]:
# ============================================================
# 6.1 ImageTextClassifier Model
# ============================================================

class ImageTextClassifier(nn.Module):
    """Multimodal classifier combining BiomedCLIP (image) and BioClinicalBERT (text).
    
    Architecture:
        Image → BiomedCLIP ViT → 512-d embedding
        Text  → BioClinicalBERT → 768-d CLS embedding
        Concat → [1280] → Fusion MLP → [6] (disease logits)
    
    Fusion Head:
        Linear(1280, 512) → LayerNorm → GELU → Dropout(0.3)
        Linear(512, 256)  → LayerNorm → GELU → Dropout(0.3)
        Linear(256, 6)
    """

    def __init__(
        self,
        biomed_clip_model_name: str,
        text_model_name: str,
        num_classes: int = 6,
        image_embed_dim: int = 512,
        text_embed_dim: int = 768,
        freeze_image_blocks: int = 6,
        freeze_text_layers: int = 10,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.image_embed_dim = image_embed_dim
        self.text_embed_dim = text_embed_dim

        # ── Image Encoder (BiomedCLIP ViT) ─────────────────────
        clip_model, _ = open_clip.create_model_from_pretrained(biomed_clip_model_name)
        self.image_encoder = clip_model.visual  # Extract the visual (ViT) component
        del clip_model
        gc.collect()

        # Freeze first N ViT transformer blocks
        self._freeze_image_encoder(freeze_image_blocks)

        # ── Text Encoder (BioClinicalBERT) ─────────────────────
        self.text_encoder = AutoModel.from_pretrained(text_model_name)

        # Freeze first N BERT layers
        self._freeze_text_encoder(freeze_text_layers)

        # ── Fusion Head ────────────────────────────────────────
        fused_dim = image_embed_dim + text_embed_dim  # 1280
        self.fusion_head = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

        # Xavier initialization for fusion head
        self._init_fusion_head()

    def _freeze_image_encoder(self, freeze_blocks: int):
        """Freeze the first `freeze_blocks` ViT transformer blocks."""
        # Freeze patch embedding and position embedding
        for param in self.image_encoder.conv1.parameters():
            param.requires_grad = False
        if hasattr(self.image_encoder, 'positional_embedding'):
            if self.image_encoder.positional_embedding is not None:
                self.image_encoder.positional_embedding.requires_grad = False
        if hasattr(self.image_encoder, 'class_embedding'):
            if self.image_encoder.class_embedding is not None:
                self.image_encoder.class_embedding.requires_grad = False

        # Freeze transformer blocks
        if hasattr(self.image_encoder, 'transformer'):
            blocks = self.image_encoder.transformer.resblocks
        elif hasattr(self.image_encoder, 'trunk'):
            blocks = self.image_encoder.trunk.blocks
        else:
            raise ValueError("Could not find ViT transformer blocks in image encoder.")

        total_blocks = len(blocks)
        print(f"  🧊 Image encoder: freezing {freeze_blocks}/{total_blocks} blocks")
        for i, block in enumerate(blocks):
            if i < freeze_blocks:
                for param in block.parameters():
                    param.requires_grad = False

    def _freeze_text_encoder(self, freeze_layers: int):
        """Freeze the first `freeze_layers` BERT layers, keep last layers + pooler trainable."""
        # Freeze embeddings
        for param in self.text_encoder.embeddings.parameters():
            param.requires_grad = False

        total_layers = len(self.text_encoder.encoder.layer)
        print(f"  🧊 Text encoder: freezing {freeze_layers}/{total_layers} layers")
        for i, layer in enumerate(self.text_encoder.encoder.layer):
            if i < freeze_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        # Ensure pooler is trainable
        if hasattr(self.text_encoder, 'pooler') and self.text_encoder.pooler is not None:
            for param in self.text_encoder.pooler.parameters():
                param.requires_grad = True

    def _init_fusion_head(self):
        """Xavier initialization for all linear layers in the fusion head."""
        for module in self.fusion_head:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def get_image_embedding(self, image: torch.Tensor) -> torch.Tensor:
        """Extract image embedding from BiomedCLIP ViT.
        
        Args:
            image: [B, 3, 224, 224]
        Returns:
            embedding: [B, 512]
        """
        return self.image_encoder(image)

    def get_text_embedding(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """Extract CLS token embedding from BioClinicalBERT.
        
        Args:
            input_ids: [B, max_text_len]
            attention_mask: [B, max_text_len]
        Returns:
            embedding: [B, 768]
        """
        outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        # Use CLS token (index 0) from last hidden state
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [B, 768]
        return cls_embedding

    def forward(
        self,
        image: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """Forward pass through the multimodal classifier.
        
        Returns:
            dict with keys:
                - logits: [B, num_classes]
                - image_embed: [B, 512]
                - text_embed: [B, 768]
                - fused_embed: [B, 1280]
        """
        # Extract modality embeddings
        image_embed = self.get_image_embedding(image)                          # [B, 512]
        text_embed = self.get_text_embedding(input_ids, attention_mask)         # [B, 768]

        # Late fusion: concatenation
        fused_embed = torch.cat([image_embed, text_embed], dim=1)              # [B, 1280]

        # Classification
        logits = self.fusion_head(fused_embed)                                 # [B, 6]

        return {
            "logits": logits,
            "image_embed": image_embed,
            "text_embed": text_embed,
            "fused_embed": fused_embed,
        }


print("✅ ImageTextClassifier class defined.")

In [ ]:
# ============================================================
# 6.2 Instantiate Model
# ============================================================

model = ImageTextClassifier(
    biomed_clip_model_name=CONFIG["biomed_clip_model"],
    text_model_name=CONFIG["text_model_name"],
    num_classes=CONFIG["num_classes"],
    image_embed_dim=CONFIG["image_embed_dim"],
    text_embed_dim=CONFIG["text_embed_dim"],
    freeze_image_blocks=CONFIG["biomed_clip_freeze_blocks"],
    freeze_text_layers=CONFIG["bert_freeze_layers"],
)

# Multi-GPU support
if NUM_GPUS > 1:
    print(f"🔀 Using nn.DataParallel across {NUM_GPUS} GPUs.")
    model = nn.DataParallel(model)

model = model.to(DEVICE)

# Print parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\n📊 Model Parameters:")
print(f"   Total:     {total_params:>12,}")
print(f"   Trainable: {trainable_params:>12,} ({100*trainable_params/total_params:.1f}%)")
print(f"   Frozen:    {frozen_params:>12,} ({100*frozen_params/total_params:.1f}%)")

---
## 7. Loss Function (Weighted Focal Loss)

Focal Loss addresses class imbalance by down-weighting easy examples and focusing on hard ones. We compute class weights from the training set and clip them to a safe range.

In [ ]:
# ============================================================
# 7.1 Compute Class Weights
# ============================================================

def compute_class_weights(
    df: pd.DataFrame,
    labels: List[str],
    clip_min: float = 0.5,
    clip_max: float = 20.0,
) -> torch.Tensor:
    """Compute inverse-frequency class weights, clipped to a safe range.
    
    Weight_i = N / (num_classes * count_i)
    """
    n_samples = len(df)
    n_classes = len(labels)
    weights = []
    for label in labels:
        count = df[label].sum()
        if count == 0:
            w = clip_max
        else:
            w = n_samples / (n_classes * count)
        w = max(clip_min, min(clip_max, w))
        weights.append(w)
    return torch.tensor(weights, dtype=torch.float32)


class_weights = compute_class_weights(
    df_train,
    DISEASE_LABELS,
    clip_min=CONFIG["weight_clip_min"],
    clip_max=CONFIG["weight_clip_max"],
).to(DEVICE)

print("⚖️  Class weights:")
for label, w in zip(DISEASE_LABELS, class_weights):
    print(f"   {label:<20s}: {w:.4f}")

In [ ]:
# ============================================================
# 7.2 Focal Loss Implementation
# ============================================================

class WeightedFocalLoss(nn.Module):
    """Weighted Focal Loss for multi-label classification.
    
    Focal Loss = -alpha * (1 - p_t)^gamma * log(p_t)
    
    where p_t is the predicted probability of the true class.
    
    Args:
        gamma: Focusing parameter (default=2.0)
        class_weights: Per-class weights tensor [num_classes]
    """

    def __init__(self, gamma: float = 2.0, class_weights: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("class_weights", class_weights)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """Compute weighted focal loss.
        
        Args:
            logits:  [B, C] raw logits
            targets: [B, C] binary targets
        Returns:
            Scalar loss
        """
        probs = torch.sigmoid(logits)
        # BCE component
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        # Focal modulating factor
        p_t = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1 - p_t) ** self.gamma
        # Apply focal weight
        loss = focal_weight * bce
        # Apply class weights
        if self.class_weights is not None:
            loss = loss * self.class_weights.unsqueeze(0)
        return loss.mean()


criterion = WeightedFocalLoss(
    gamma=CONFIG["focal_gamma"],
    class_weights=class_weights,
)

print(f"✅ Weighted Focal Loss created (gamma={CONFIG['focal_gamma']})")

---
## 8. Optimizer, Scheduler, EMA

In [ ]:
# ============================================================
# 8.1 Optimizer with Differential Learning Rates
# ============================================================

def get_optimizer(model, config: dict) -> AdamW:
    """Create AdamW optimizer with three learning rate groups.
    
    Groups:
        1. Image encoder (trainable params) → lr_image_encoder
        2. Text encoder (trainable params)  → lr_text_encoder
        3. Fusion head                      → lr_fusion_head
    """
    # Handle DataParallel wrapper
    base_model = model.module if hasattr(model, 'module') else model

    image_params = [
        p for p in base_model.image_encoder.parameters() if p.requires_grad
    ]
    text_params = [
        p for p in base_model.text_encoder.parameters() if p.requires_grad
    ]
    fusion_params = [
        p for p in base_model.fusion_head.parameters() if p.requires_grad
    ]

    param_groups = [
        {"params": image_params, "lr": config["lr_image_encoder"],  "name": "image_encoder"},
        {"params": text_params,  "lr": config["lr_text_encoder"],   "name": "text_encoder"},
        {"params": fusion_params, "lr": config["lr_fusion_head"],   "name": "fusion_head"},
    ]

    optimizer = AdamW(param_groups, weight_decay=config["weight_decay"])

    print("\n🔧 Optimizer parameter groups:")
    for g in param_groups:
        n_params = sum(p.numel() for p in g["params"])
        print(f"   {g['name']:<16s}: {n_params:>10,} params, lr={g['lr']:.1e}")

    return optimizer


optimizer = get_optimizer(model, CONFIG)

In [ ]:
# ============================================================
# 8.2 Learning Rate Scheduler
# ============================================================

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
    eta_min=1e-7,
)
print(f"✅ CosineAnnealingLR scheduler created (T_max={CONFIG['epochs']})")

In [ ]:
# ============================================================
# 8.3 Exponential Moving Average (EMA)
# ============================================================

class EMA:
    """Exponential Moving Average of model parameters.
    
    Maintains a shadow copy of model parameters that is updated
    as: shadow = decay * shadow + (1 - decay) * param
    
    The EMA model typically generalizes better than the final model.
    """

    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        # Initialize shadow parameters
        base_model = model.module if hasattr(model, 'module') else model
        for name, param in base_model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    @torch.no_grad()
    def update(self, model: nn.Module):
        """Update shadow parameters after each optimization step."""
        base_model = model.module if hasattr(model, 'module') else model
        for name, param in base_model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = (
                    self.decay * self.shadow[name] + (1.0 - self.decay) * param.data
                )

    def apply_shadow(self, model: nn.Module):
        """Replace model parameters with EMA shadow parameters."""
        base_model = model.module if hasattr(model, 'module') else model
        self.backup = {}
        for name, param in base_model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name].clone()

    def restore(self, model: nn.Module):
        """Restore model parameters from backup (undo apply_shadow)."""
        base_model = model.module if hasattr(model, 'module') else model
        for name, param in base_model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name].clone()
        self.backup = {}

    def state_dict(self):
        return {"shadow": self.shadow, "decay": self.decay}

    def load_state_dict(self, state_dict):
        self.shadow = state_dict["shadow"]
        self.decay = state_dict["decay"]


ema = EMA(model, decay=CONFIG["ema_decay"])
print(f"✅ EMA initialized (decay={CONFIG['ema_decay']})")

In [ ]:
# ============================================================
# 8.4 Mixed Precision Scaler
# ============================================================

scaler = GradScaler(enabled=CONFIG["mixed_precision"])
print(f"✅ GradScaler created (enabled={CONFIG['mixed_precision']})")

---
## 9. Training Loop

The training loop includes:
- AMP mixed precision
- Gradient clipping
- EMA updates
- Early stopping
- Checkpoint saving
- Resume support

In [ ]:
# ============================================================
# 9.1 Training & Validation Step Functions
# ============================================================

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    ema: EMA,
    device: torch.device,
    max_grad_norm: float = 1.0,
    epoch: int = 0,
) -> Dict[str, float]:
    """Train for one epoch with AMP, gradient clipping, and EMA."""
    model.train()
    running_loss = 0.0
    all_logits = []
    all_labels = []
    n_batches = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [Train]", leave=True)

    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Forward pass with AMP
        with autocast(enabled=CONFIG["mixed_precision"]):
            outputs = model(images, input_ids, attention_mask)
            loss = criterion(outputs["logits"], labels)

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        # Optimizer step
        scaler.step(optimizer)
        scaler.update()

        # EMA update
        ema.update(model)

        # Accumulate metrics
        running_loss += loss.item()
        n_batches += 1
        all_logits.append(outputs["logits"].detach().cpu())
        all_labels.append(labels.detach().cpu())

        pbar.set_postfix({"loss": f"{running_loss/n_batches:.4f}"})

    avg_loss = running_loss / max(n_batches, 1)
    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    all_probs = torch.sigmoid(all_logits).numpy()
    all_labels_np = all_labels.numpy()

    # Compute AUROC
    try:
        auroc = roc_auc_score(all_labels_np, all_probs, average="macro")
    except ValueError:
        auroc = 0.0

    return {"loss": avg_loss, "auroc": auroc}


@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    epoch: int = 0,
) -> Dict[str, Any]:
    """Validate the model and return loss, AUROC, and predictions."""
    model.eval()
    running_loss = 0.0
    all_logits = []
    all_labels = []
    all_study_ids = []
    n_batches = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [Val]", leave=True)

    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with autocast(enabled=CONFIG["mixed_precision"]):
            outputs = model(images, input_ids, attention_mask)
            loss = criterion(outputs["logits"], labels)

        running_loss += loss.item()
        n_batches += 1
        all_logits.append(outputs["logits"].cpu())
        all_labels.append(labels.cpu())
        all_study_ids.extend(batch["study_id"].tolist())

        pbar.set_postfix({"loss": f"{running_loss/n_batches:.4f}"})

    avg_loss = running_loss / max(n_batches, 1)
    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    all_probs = torch.sigmoid(all_logits).numpy()
    all_labels_np = all_labels.numpy()

    try:
        auroc = roc_auc_score(all_labels_np, all_probs, average="macro")
    except ValueError:
        auroc = 0.0

    return {
        "loss": avg_loss,
        "auroc": auroc,
        "probs": all_probs,
        "labels": all_labels_np,
        "study_ids": all_study_ids,
    }


print("✅ Training and validation functions defined.")

In [ ]:
# ============================================================
# 9.2 Checkpoint Utilities
# ============================================================

def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler: GradScaler,
    ema: EMA,
    epoch: int,
    best_auroc: float,
    history: dict,
    filepath: str,
):
    """Save a training checkpoint for resume support."""
    base_model = model.module if hasattr(model, 'module') else model
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": base_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "ema_state_dict": ema.state_dict(),
        "best_auroc": best_auroc,
        "history": history,
        "config": CONFIG,
    }
    torch.save(checkpoint, filepath)


def load_checkpoint(filepath: str, model, optimizer, scheduler, scaler, ema):
    """Load a training checkpoint for resume."""
    checkpoint = torch.load(filepath, map_location=DEVICE)
    base_model = model.module if hasattr(model, 'module') else model
    base_model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    ema.load_state_dict(checkpoint["ema_state_dict"])
    return checkpoint["epoch"], checkpoint["best_auroc"], checkpoint["history"]


print("✅ Checkpoint utilities defined.")

In [ ]:
# ============================================================
# 9.3 Main Training Loop
# ============================================================

# Resume support
RESUME_PATH = os.path.join(CONFIG["output_dir"], "checkpoint_last.pth")
start_epoch = 0
best_auroc = 0.0
patience_counter = 0

history = {
    "train_loss": [],
    "train_auroc": [],
    "val_loss": [],
    "val_auroc": [],
    "lr": [],
    "epoch_time": [],
}

# Check for existing checkpoint to resume
if os.path.exists(RESUME_PATH):
    print(f"📂 Resuming from checkpoint: {RESUME_PATH}")
    start_epoch, best_auroc, history = load_checkpoint(
        RESUME_PATH, model, optimizer, scheduler, scaler, ema
    )
    start_epoch += 1  # Start from next epoch
    patience_counter = 0
    print(f"   Resuming from epoch {start_epoch}, best AUROC: {best_auroc:.4f}")
else:
    print("🆕 Starting training from scratch.")

print(f"\n{'='*70}")
print(f"🚀 TRAINING START")
print(f"   Epochs: {start_epoch} → {CONFIG['epochs']}")
print(f"   Patience: {CONFIG['patience']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"{'='*70}\n")

for epoch in range(start_epoch, CONFIG["epochs"]):
    epoch_start = time.time()

    # ── Train ──────────────────────────────────────────────────
    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        ema=ema,
        device=DEVICE,
        max_grad_norm=CONFIG["max_grad_norm"],
        epoch=epoch,
    )

    # ── Validate ───────────────────────────────────────────────
    val_metrics = validate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        epoch=epoch,
    )

    # ── Scheduler step ─────────────────────────────────────────
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # ── Record history ─────────────────────────────────────────
    epoch_time = time.time() - epoch_start
    history["train_loss"].append(train_metrics["loss"])
    history["train_auroc"].append(train_metrics["auroc"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_auroc"].append(val_metrics["auroc"])
    history["lr"].append(current_lr)
    history["epoch_time"].append(epoch_time)

    # ── Print summary ──────────────────────────────────────────
    print(
        f"\n📊 Epoch {epoch+1}/{CONFIG['epochs']} "
        f"| Train Loss: {train_metrics['loss']:.4f} "
        f"| Train AUROC: {train_metrics['auroc']:.4f} "
        f"| Val Loss: {val_metrics['loss']:.4f} "
        f"| Val AUROC: {val_metrics['auroc']:.4f} "
        f"| LR: {current_lr:.2e} "
        f"| Time: {epoch_time:.1f}s"
    )

    # ── Best model saving ──────────────────────────────────────
    if val_metrics["auroc"] > best_auroc:
        best_auroc = val_metrics["auroc"]
        patience_counter = 0
        # Save best model
        base_model = model.module if hasattr(model, 'module') else model
        torch.save(
            base_model.state_dict(),
            os.path.join(CONFIG["output_dir"], "best_model.pth"),
        )
        # Save EMA model
        ema.apply_shadow(model)
        torch.save(
            base_model.state_dict(),
            os.path.join(CONFIG["output_dir"], "best_model_ema.pth"),
        )
        ema.restore(model)
        print(f"   🏆 New best AUROC: {best_auroc:.4f} — Model saved!")
    else:
        patience_counter += 1
        print(f"   ⏳ Patience: {patience_counter}/{CONFIG['patience']}")

    # ── Save last checkpoint ───────────────────────────────────
    save_checkpoint(
        model, optimizer, scheduler, scaler, ema,
        epoch, best_auroc, history, RESUME_PATH,
    )

    # ── Early stopping ─────────────────────────────────────────
    if patience_counter >= CONFIG["patience"]:
        print(f"\n🛑 Early stopping triggered at epoch {epoch+1}.")
        break

print(f"\n{'='*70}")
print(f"✅ TRAINING COMPLETE")
print(f"   Best Validation AUROC: {best_auroc:.4f}")
print(f"   Total training time: {sum(history['epoch_time']):.0f}s")
print(f"{'='*70}")

In [ ]:
# ============================================================
# 9.4 Save Training History
# ============================================================

with open(os.path.join(CONFIG["output_dir"], "training_history.pkl"), "wb") as f:
    pickle.dump(history, f)
print("✅ Training history saved.")

---
## 10. Comprehensive Evaluation

We compute per-disease and aggregate metrics using the best model, EMA model, and EMA + TTA.

In [ ]:
# ============================================================
# 10.1 Metric Computation Functions
# ============================================================

def compute_all_metrics(
    labels: np.ndarray,
    probs: np.ndarray,
    disease_names: List[str],
    thresholds: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """Compute comprehensive multi-label classification metrics.
    
    Args:
        labels: [N, C] ground truth binary labels
        probs:  [N, C] predicted probabilities
        disease_names: list of disease names
        thresholds: [C] per-class thresholds (default 0.5)
    
    Returns:
        Dictionary with per-disease and aggregate metrics.
    """
    n_classes = labels.shape[1]
    if thresholds is None:
        thresholds = np.full(n_classes, 0.5)

    preds = (probs >= thresholds).astype(int)

    metrics = {"per_disease": {}, "aggregate": {}}

    # ── Per-disease metrics ────────────────────────────────────
    for i, name in enumerate(disease_names):
        disease_metrics = {}
        try:
            disease_metrics["auroc"] = float(roc_auc_score(labels[:, i], probs[:, i]))
        except ValueError:
            disease_metrics["auroc"] = 0.0

        disease_metrics["f1"] = float(f1_score(labels[:, i], preds[:, i], zero_division=0))
        disease_metrics["accuracy"] = float(accuracy_score(labels[:, i], preds[:, i]))

        try:
            disease_metrics["auprc"] = float(average_precision_score(labels[:, i], probs[:, i]))
        except ValueError:
            disease_metrics["auprc"] = 0.0

        disease_metrics["threshold"] = float(thresholds[i])
        metrics["per_disease"][name] = disease_metrics

    # ── Aggregate metrics ──────────────────────────────────────
    try:
        metrics["aggregate"]["macro_auroc"] = float(
            roc_auc_score(labels, probs, average="macro")
        )
        metrics["aggregate"]["micro_auroc"] = float(
            roc_auc_score(labels, probs, average="micro")
        )
    except ValueError:
        metrics["aggregate"]["macro_auroc"] = 0.0
        metrics["aggregate"]["micro_auroc"] = 0.0

    metrics["aggregate"]["macro_f1"] = float(
        f1_score(labels, preds, average="macro", zero_division=0)
    )
    metrics["aggregate"]["micro_f1"] = float(
        f1_score(labels, preds, average="micro", zero_division=0)
    )
    metrics["aggregate"]["macro_precision"] = float(
        precision_score(labels, preds, average="macro", zero_division=0)
    )
    metrics["aggregate"]["macro_recall"] = float(
        recall_score(labels, preds, average="macro", zero_division=0)
    )

    try:
        metrics["aggregate"]["macro_ap"] = float(
            average_precision_score(labels, probs, average="macro")
        )
    except ValueError:
        metrics["aggregate"]["macro_ap"] = 0.0

    # Exact match accuracy (all labels correct)
    metrics["aggregate"]["exact_match_accuracy"] = float(
        np.all(preds == labels, axis=1).mean()
    )

    # Hamming accuracy (1 - hamming_loss)
    metrics["aggregate"]["hamming_accuracy"] = float(
        1.0 - hamming_loss(labels, preds)
    )

    # Mean per-class accuracy
    per_class_acc = [
        metrics["per_disease"][name]["accuracy"] for name in disease_names
    ]
    metrics["aggregate"]["mean_per_class_accuracy"] = float(np.mean(per_class_acc))

    return metrics


def print_metrics(metrics: Dict[str, Any], title: str = "Metrics"):
    """Pretty-print computed metrics."""
    print(f"\n{'='*70}")
    print(f"📊 {title}")
    print(f"{'='*70}")

    print(f"\n{'─'*70}")
    print(f"{'Disease':<20s} {'AUROC':>8s} {'F1':>8s} {'Acc':>8s} {'AUPRC':>8s} {'Thresh':>8s}")
    print(f"{'─'*70}")
    for name, m in metrics["per_disease"].items():
        print(
            f"{name:<20s} {m['auroc']:>8.4f} {m['f1']:>8.4f} "
            f"{m['accuracy']:>8.4f} {m['auprc']:>8.4f} {m['threshold']:>8.3f}"
        )

    print(f"\n{'─'*70}")
    print("Aggregate Metrics:")
    print(f"{'─'*70}")
    for key, val in metrics["aggregate"].items():
        print(f"  {key:<30s}: {val:.4f}")


print("✅ Metric functions defined.")

In [ ]:
# ============================================================
# 10.2 Threshold Optimization
# ============================================================

def optimize_thresholds(
    labels: np.ndarray,
    probs: np.ndarray,
    disease_names: List[str],
    start: float = 0.10,
    end: float = 0.90,
    steps: int = 81,
) -> Tuple[np.ndarray, Dict[str, List]]:
    """Find optimal per-class thresholds that maximize F1 score.
    
    Searches over `steps` candidate thresholds in [start, end].
    
    Returns:
        best_thresholds: [num_classes] optimal thresholds
        threshold_curves: dict mapping disease name → list of (threshold, f1) tuples
    """
    candidates = np.linspace(start, end, steps)
    n_classes = labels.shape[1]
    best_thresholds = np.full(n_classes, 0.5)
    threshold_curves = {}

    print(f"\n🔍 Optimizing thresholds ({steps} candidates in [{start:.2f}, {end:.2f}])")

    for i, name in enumerate(disease_names):
        best_f1 = 0.0
        best_t = 0.5
        curve = []

        for t in candidates:
            preds = (probs[:, i] >= t).astype(int)
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            curve.append((float(t), float(f1)))
            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds[i] = best_t
        threshold_curves[name] = curve
        print(f"  {name:<20s}: threshold={best_t:.3f}, F1={best_f1:.4f}")

    return best_thresholds, threshold_curves


print("✅ Threshold optimization function defined.")

In [ ]:
# ============================================================
# 10.3 Load Best Model & Evaluate (Baseline)
# ============================================================

# Load best model weights
base_model = model.module if hasattr(model, 'module') else model
best_weights = torch.load(
    os.path.join(CONFIG["output_dir"], "best_model.pth"),
    map_location=DEVICE,
)
base_model.load_state_dict(best_weights)
model.to(DEVICE)
print("✅ Loaded best model weights.")

# Validate
val_results = validate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=DEVICE,
    epoch=0,
)

# Optimize thresholds
optimal_thresholds, threshold_curves = optimize_thresholds(
    labels=val_results["labels"],
    probs=val_results["probs"],
    disease_names=DISEASE_LABELS,
    start=CONFIG["threshold_start"],
    end=CONFIG["threshold_end"],
    steps=CONFIG["threshold_steps"],
)

# Compute final metrics with optimal thresholds
baseline_metrics = compute_all_metrics(
    labels=val_results["labels"],
    probs=val_results["probs"],
    disease_names=DISEASE_LABELS,
    thresholds=optimal_thresholds,
)
print_metrics(baseline_metrics, "Baseline Model (Optimal Thresholds)")

# Save thresholds
thresholds_dict = {
    name: float(t) for name, t in zip(DISEASE_LABELS, optimal_thresholds)
}
with open(os.path.join(CONFIG["output_dir"], "thresholds.json"), "w") as f:
    json.dump(thresholds_dict, f, indent=2)
print("\n✅ Thresholds saved to thresholds.json")

In [ ]:
# ============================================================
# 10.4 EMA Model Evaluation
# ============================================================

# Load EMA model weights
ema_weights = torch.load(
    os.path.join(CONFIG["output_dir"], "best_model_ema.pth"),
    map_location=DEVICE,
)
base_model.load_state_dict(ema_weights)
model.to(DEVICE)
print("✅ Loaded EMA model weights.")

# Validate EMA model
ema_results = validate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=DEVICE,
    epoch=0,
)

# Optimize thresholds for EMA
ema_thresholds, _ = optimize_thresholds(
    labels=ema_results["labels"],
    probs=ema_results["probs"],
    disease_names=DISEASE_LABELS,
    start=CONFIG["threshold_start"],
    end=CONFIG["threshold_end"],
    steps=CONFIG["threshold_steps"],
)

ema_metrics = compute_all_metrics(
    labels=ema_results["labels"],
    probs=ema_results["probs"],
    disease_names=DISEASE_LABELS,
    thresholds=ema_thresholds,
)
print_metrics(ema_metrics, "EMA Model (Optimal Thresholds)")

In [ ]:
# ============================================================
# 10.5 EMA + TTA Evaluation
# ============================================================

@torch.no_grad()
def evaluate_with_tta(
    model: nn.Module,
    dataset: MultimodalDataset,
    device: torch.device,
    batch_size: int = 32,
    num_workers: int = 4,
) -> Tuple[np.ndarray, np.ndarray, List[int]]:
    """Evaluate with Test-Time Augmentation.
    
    TTA strategy:
        1. Original image
        2. Horizontally flipped image
    
    Final prediction = average of both passes.
    """
    model.eval()

    # --- Pass 1: Original (no augmentation) ---
    original_transforms = get_val_transforms(CONFIG["image_size"])
    dataset_original = MultimodalDataset(
        dataframe=dataset.df,
        tokenizer=dataset.tokenizer,
        transforms=original_transforms,
        disease_labels=dataset.disease_labels,
        max_text_len=dataset.max_text_len,
    )
    loader_original = DataLoader(
        dataset_original, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )

    all_probs_original = []
    all_labels = []
    all_study_ids = []

    for batch in tqdm(loader_original, desc="TTA Pass 1 (Original)"):
        images = batch["image"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)

        with autocast(enabled=CONFIG["mixed_precision"]):
            outputs = model(images, input_ids, attention_mask)
        probs = torch.sigmoid(outputs["logits"]).cpu().numpy()
        all_probs_original.append(probs)
        all_labels.append(batch["labels"].numpy())
        all_study_ids.extend(batch["study_id"].tolist())

    # --- Pass 2: Horizontal Flip ---
    flip_transforms = T.Compose([
        T.Resize((CONFIG["image_size"], CONFIG["image_size"])),
        T.RandomHorizontalFlip(p=1.0),  # Always flip
        T.ToTensor(),
        T.Normalize(mean=BIOMED_CLIP_MEAN, std=BIOMED_CLIP_STD),
    ])
    dataset_flip = MultimodalDataset(
        dataframe=dataset.df,
        tokenizer=dataset.tokenizer,
        transforms=flip_transforms,
        disease_labels=dataset.disease_labels,
        max_text_len=dataset.max_text_len,
    )
    loader_flip = DataLoader(
        dataset_flip, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )

    all_probs_flip = []
    for batch in tqdm(loader_flip, desc="TTA Pass 2 (H-Flip)"):
        images = batch["image"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)

        with autocast(enabled=CONFIG["mixed_precision"]):
            outputs = model(images, input_ids, attention_mask)
        probs = torch.sigmoid(outputs["logits"]).cpu().numpy()
        all_probs_flip.append(probs)

    # --- Average ---
    probs_original = np.concatenate(all_probs_original, axis=0)
    probs_flip = np.concatenate(all_probs_flip, axis=0)
    probs_avg = (probs_original + probs_flip) / 2.0
    labels_all = np.concatenate(all_labels, axis=0)

    return probs_avg, labels_all, all_study_ids


print("\n🔄 Running EMA + TTA evaluation...")
tta_probs, tta_labels, tta_study_ids = evaluate_with_tta(
    model=model,
    dataset=val_dataset,
    device=DEVICE,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
)

# Optimize thresholds for TTA
tta_thresholds, _ = optimize_thresholds(
    labels=tta_labels,
    probs=tta_probs,
    disease_names=DISEASE_LABELS,
    start=CONFIG["threshold_start"],
    end=CONFIG["threshold_end"],
    steps=CONFIG["threshold_steps"],
)

tta_metrics = compute_all_metrics(
    labels=tta_labels,
    probs=tta_probs,
    disease_names=DISEASE_LABELS,
    thresholds=tta_thresholds,
)
print_metrics(tta_metrics, "EMA + TTA (Optimal Thresholds)")

In [ ]:
# ============================================================
# 10.6 Comparison Summary
# ============================================================

print(f"\n{'='*70}")
print(f"📊 MODEL COMPARISON SUMMARY")
print(f"{'='*70}")
print(f"{'Method':<25s} {'Macro AUROC':>12s} {'Macro F1':>10s} {'Exact Match':>12s}")
print(f"{'─'*70}")
print(
    f"{'Baseline':<25s} "
    f"{baseline_metrics['aggregate']['macro_auroc']:>12.4f} "
    f"{baseline_metrics['aggregate']['macro_f1']:>10.4f} "
    f"{baseline_metrics['aggregate']['exact_match_accuracy']:>12.4f}"
)
print(
    f"{'EMA':<25s} "
    f"{ema_metrics['aggregate']['macro_auroc']:>12.4f} "
    f"{ema_metrics['aggregate']['macro_f1']:>10.4f} "
    f"{ema_metrics['aggregate']['exact_match_accuracy']:>12.4f}"
)
print(
    f"{'EMA + TTA':<25s} "
    f"{tta_metrics['aggregate']['macro_auroc']:>12.4f} "
    f"{tta_metrics['aggregate']['macro_f1']:>10.4f} "
    f"{tta_metrics['aggregate']['exact_match_accuracy']:>12.4f}"
)

---
## 11. Visualizations

In [ ]:
# ============================================================
# 11.1 Training History Plot
# ============================================================

def plot_training_history(history: dict, save_path: str):
    """Plot training loss, AUROC, and learning rate curves."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    epochs = range(1, len(history["train_loss"]) + 1)

    # Loss
    axes[0].plot(epochs, history["train_loss"], "b-o", label="Train", markersize=4)
    axes[0].plot(epochs, history["val_loss"], "r-o", label="Validation", markersize=4)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training & Validation Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # AUROC
    axes[1].plot(epochs, history["train_auroc"], "b-o", label="Train", markersize=4)
    axes[1].plot(epochs, history["val_auroc"], "r-o", label="Validation", markersize=4)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("AUROC")
    axes[1].set_title("Training & Validation AUROC")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Learning rate
    axes[2].plot(epochs, history["lr"], "g-o", markersize=4)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Learning Rate")
    axes[2].set_title("Learning Rate Schedule")
    axes[2].set_yscale("log")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")


plot_training_history(
    history,
    os.path.join(CONFIG["output_dir"], "training_history.png"),
)

In [ ]:
# ============================================================
# 11.2 ROC Curves
# ============================================================

def plot_roc_curves(
    labels: np.ndarray,
    probs: np.ndarray,
    disease_names: List[str],
    save_path: str,
):
    """Plot per-disease ROC curves."""
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.Set2(np.linspace(0, 1, len(disease_names)))

    for i, (name, color) in enumerate(zip(disease_names, colors)):
        try:
            fpr, tpr, _ = roc_curve(labels[:, i], probs[:, i])
            auc_val = roc_auc_score(labels[:, i], probs[:, i])
            ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc_val:.3f})")
        except ValueError:
            pass

    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curves (Per Disease)", fontsize=14)
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")


# Use EMA results for plotting
plot_roc_curves(
    ema_results["labels"],
    ema_results["probs"],
    DISEASE_LABELS,
    os.path.join(CONFIG["output_dir"], "roc_curves.png"),
)

In [ ]:
# ============================================================
# 11.3 Confusion Matrices
# ============================================================

def plot_confusion_matrices(
    labels: np.ndarray,
    probs: np.ndarray,
    thresholds: np.ndarray,
    disease_names: List[str],
    save_path: str,
):
    """Plot per-disease confusion matrices."""
    preds = (probs >= thresholds).astype(int)
    n_classes = len(disease_names)
    n_cols = 3
    n_rows = math.ceil(n_classes / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    for i, name in enumerate(disease_names):
        cm = confusion_matrix(labels[:, i], preds[:, i], labels=[0, 1])
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Neg", "Pos"],
            yticklabels=["Neg", "Pos"],
            ax=axes[i],
        )
        axes[i].set_title(f"{name}\n(thresh={thresholds[i]:.3f})")
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("Actual")

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Confusion Matrices (Per Disease)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")


plot_confusion_matrices(
    ema_results["labels"],
    ema_results["probs"],
    ema_thresholds,
    DISEASE_LABELS,
    os.path.join(CONFIG["output_dir"], "confusion_matrices.png"),
)

In [ ]:
# ============================================================
# 11.4 AUROC Bar Chart
# ============================================================

def plot_auroc_bar(
    metrics: Dict[str, Any],
    disease_names: List[str],
    save_path: str,
):
    """Plot per-disease AUROC as a bar chart."""
    aurocs = [metrics["per_disease"][name]["auroc"] for name in disease_names]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(disease_names)))

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(disease_names, aurocs, color=colors, edgecolor="black", linewidth=0.5)

    # Add value labels
    for bar, val in zip(bars, aurocs):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{val:.3f}",
            ha="center", va="bottom", fontsize=11, fontweight="bold",
        )

    # Add macro AUROC line
    macro_auroc = metrics["aggregate"]["macro_auroc"]
    ax.axhline(y=macro_auroc, color="red", linestyle="--", linewidth=2,
               label=f"Macro AUROC: {macro_auroc:.3f}")

    ax.set_ylabel("AUROC", fontsize=12)
    ax.set_title("Per-Disease AUROC", fontsize=14)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, axis="y")
    plt.xticks(rotation=15, ha="right")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")


plot_auroc_bar(
    ema_metrics,
    DISEASE_LABELS,
    os.path.join(CONFIG["output_dir"], "auroc_bar.png"),
)

In [ ]:
# ============================================================
# 11.5 Threshold Optimization Curves
# ============================================================

def plot_threshold_curves(
    threshold_curves: Dict[str, List],
    optimal_thresholds: np.ndarray,
    disease_names: List[str],
    save_path: str,
):
    """Plot F1 score vs threshold for each disease."""
    n_classes = len(disease_names)
    n_cols = 3
    n_rows = math.ceil(n_classes / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    for i, name in enumerate(disease_names):
        curve = threshold_curves[name]
        thresholds_list = [c[0] for c in curve]
        f1_list = [c[1] for c in curve]

        axes[i].plot(thresholds_list, f1_list, "b-", linewidth=2)
        axes[i].axvline(
            x=optimal_thresholds[i], color="r", linestyle="--",
            label=f"Optimal: {optimal_thresholds[i]:.3f}",
        )
        axes[i].set_xlabel("Threshold")
        axes[i].set_ylabel("F1 Score")
        axes[i].set_title(name)
        axes[i].legend(fontsize=9)
        axes[i].grid(True, alpha=0.3)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Threshold vs F1 Score", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {save_path}")


plot_threshold_curves(
    threshold_curves,
    optimal_thresholds,
    DISEASE_LABELS,
    os.path.join(CONFIG["output_dir"], "threshold_curves.png"),
)

---
## 12. Explainability

### 12.1 GradCAM for BiomedCLIP ViT

We implement GradCAM adapted for Vision Transformers to visualize which image regions drive predictions.

In [ ]:
# ============================================================
# 12.1 GradCAM for ViT
# ============================================================

class ViTGradCAM:
    """GradCAM implementation for Vision Transformer (ViT) models.
    
    Captures activations and gradients from a target ViT block,
    then computes a spatial attention map.
    
    Supports blocks[-1], blocks[-2], blocks[-3].
    """

    def __init__(self, model: nn.Module, target_block_idx: int = -1):
        """
        Args:
            model: ImageTextClassifier model
            target_block_idx: Index of the ViT block to hook (-1, -2, -3)
        """
        self.model = model
        base_model = model.module if hasattr(model, 'module') else model

        # Find the target block
        image_encoder = base_model.image_encoder
        if hasattr(image_encoder, 'transformer'):
            blocks = image_encoder.transformer.resblocks
        elif hasattr(image_encoder, 'trunk'):
            blocks = image_encoder.trunk.blocks
        else:
            raise ValueError("Cannot find ViT blocks.")

        self.target_block = blocks[target_block_idx]
        self.activations = None
        self.gradients = None

        # Register hooks
        self.target_block.register_forward_hook(self._forward_hook)
        self.target_block.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        """Capture activations from the target block."""
        if isinstance(output, tuple):
            self.activations = output[0].detach()
        else:
            self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        """Capture gradients flowing into the target block."""
        self.gradients = grad_output[0].detach()

    def generate(
        self,
        image: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        target_class: int,
    ) -> np.ndarray:
        """Generate GradCAM heatmap for a target class.
        
        Args:
            image: [1, 3, 224, 224]
            input_ids: [1, max_text_len]
            attention_mask: [1, max_text_len]
            target_class: class index (0-5)
        
        Returns:
            heatmap: [224, 224] normalized attention map
        """
        self.model.eval()
        self.model.zero_grad()

        # Enable gradients for image
        image = image.requires_grad_(True)

        # Forward pass
        outputs = self.model(image, input_ids, attention_mask)
        logits = outputs["logits"]

        # Backward pass for target class
        target = logits[0, target_class]
        target.backward(retain_graph=True)

        # Get activations and gradients
        activations = self.activations  # [B, num_tokens, D] or [num_tokens, B, D]
        gradients = self.gradients      # same shape

        # Handle different tensor orderings
        if activations.dim() == 3:
            # If shape is [num_tokens, B, D], transpose to [B, num_tokens, D]
            if activations.shape[0] != image.shape[0]:
                activations = activations.permute(1, 0, 2)
                gradients = gradients.permute(1, 0, 2)

        # Remove CLS token (first token)
        act_patches = activations[:, 1:, :]   # [B, num_patches, D]
        grad_patches = gradients[:, 1:, :]    # [B, num_patches, D]

        # Compute weights via global average pooling of gradients
        weights = grad_patches.mean(dim=1, keepdim=True)  # [B, 1, D]

        # Weighted combination of activation maps
        cam = (weights * act_patches).sum(dim=-1)  # [B, num_patches]
        cam = F.relu(cam)  # Only positive contributions

        # Reshape to spatial grid
        num_patches = cam.shape[1]
        h = w = int(math.sqrt(num_patches))
        cam = cam.view(1, 1, h, w)

        # Upsample to image size
        cam = F.interpolate(
            cam, size=(224, 224), mode="bilinear", align_corners=False
        )
        cam = cam.squeeze().cpu().numpy()

        # Normalize to [0, 1]
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)

        return cam


print("✅ ViTGradCAM class defined.")

In [ ]:
# ============================================================
# 12.2 GradCAM Visualization Examples
# ============================================================

def visualize_gradcam_examples(
    model: nn.Module,
    dataset: MultimodalDataset,
    device: torch.device,
    disease_names: List[str],
    n_examples: int = 3,
    save_dir: str = ".",
):
    """Generate and visualize GradCAM heatmaps for sample images."""
    model.eval()
    gradcam_blocks = [-1, -2, -3]

    os.makedirs(os.path.join(save_dir, "gradcam_examples"), exist_ok=True)

    for sample_idx in range(min(n_examples, len(dataset))):
        sample = dataset[sample_idx]
        image = sample["image"].unsqueeze(0).to(device)
        input_ids = sample["input_ids"].unsqueeze(0).to(device)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
        labels = sample["labels"].numpy()

        # Get prediction
        with torch.no_grad():
            outputs = model(image, input_ids, attention_mask)
            probs_sample = torch.sigmoid(outputs["logits"]).cpu().numpy()[0]

        # Find the most relevant disease (highest probability positive class)
        positive_classes = np.where(labels > 0.5)[0]
        if len(positive_classes) == 0:
            target_class = int(np.argmax(probs_sample))
        else:
            target_class = int(positive_classes[np.argmax(probs_sample[positive_classes])])

        # Original image for display
        image_path = dataset.df.iloc[sample_idx]["image_path"]
        orig_img = Image.open(image_path).convert("RGB").resize((224, 224))
        orig_img_np = np.array(orig_img) / 255.0

        fig, axes = plt.subplots(1, 1 + len(gradcam_blocks), figsize=(5 * (1 + len(gradcam_blocks)), 5))

        # Original image
        axes[0].imshow(orig_img_np)
        axes[0].set_title(f"Original\nGT: {disease_names[target_class]}\nProb: {probs_sample[target_class]:.3f}")
        axes[0].axis("off")

        # GradCAM from different blocks
        for b_idx, block_idx in enumerate(gradcam_blocks):
            try:
                gradcam = ViTGradCAM(model, target_block_idx=block_idx)
                heatmap = gradcam.generate(image, input_ids, attention_mask, target_class)

                # Overlay
                axes[b_idx + 1].imshow(orig_img_np)
                axes[b_idx + 1].imshow(heatmap, cmap="jet", alpha=0.5)
                axes[b_idx + 1].set_title(f"GradCAM Block[{block_idx}]")
                axes[b_idx + 1].axis("off")
            except Exception as e:
                axes[b_idx + 1].text(0.5, 0.5, f"Error: {str(e)[:50]}",
                                     ha="center", va="center", transform=axes[b_idx + 1].transAxes)
                axes[b_idx + 1].axis("off")

        plt.suptitle(f"Sample {sample_idx + 1} — Study ID: {sample['study_id']}", fontsize=14)
        plt.tight_layout()
        save_path = os.path.join(save_dir, "gradcam_examples", f"gradcam_sample_{sample_idx + 1}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"✅ Saved: {save_path}")


visualize_gradcam_examples(
    model=model,
    dataset=val_dataset,
    device=DEVICE,
    disease_names=DISEASE_LABELS,
    n_examples=3,
    save_dir=CONFIG["output_dir"],
)

### 12.3 Text Explainability via Integrated Gradients

In [ ]:
# ============================================================
# 12.3 Integrated Gradients for Text Attribution
# ============================================================

class TextAttributor:
    """Text attribution using Captum's Integrated Gradients.
    
    Identifies the most important words in the radiology report
    for each disease prediction.
    """

    def __init__(self, model: nn.Module, tokenizer, device: torch.device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        base_model = model.module if hasattr(model, 'module') else model
        self.base_model = base_model

    def _forward_text_only(self, input_embeds, attention_mask, image_embed):
        """Forward function that takes text embeddings as input (for IG)."""
        text_output = self.base_model.text_encoder(
            inputs_embeds=input_embeds,
            attention_mask=attention_mask,
        )
        text_embed = text_output.last_hidden_state[:, 0, :]
        fused = torch.cat([image_embed, text_embed], dim=1)
        logits = self.base_model.fusion_head(fused)
        return logits

    def attribute(
        self,
        text: str,
        image: torch.Tensor,
        target_class: int,
        n_steps: int = 50,
    ) -> List[Tuple[str, float]]:
        """Compute token-level attribution scores.
        
        Args:
            text: radiology report text
            image: preprocessed image tensor [1, 3, 224, 224]
            target_class: class index
            n_steps: number of interpolation steps for IG
        
        Returns:
            List of (token, attribution_score) tuples
        """
        self.model.eval()

        # Tokenize
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=CONFIG["max_text_len"],
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].to(self.device)
        attention_mask = encoding["attention_mask"].to(self.device)

        # Get token embeddings
        token_embeds = self.base_model.text_encoder.embeddings(
            input_ids=input_ids
        )  # [1, seq_len, 768]
        token_embeds = token_embeds.detach().requires_grad_(True)

        # Get fixed image embedding
        with torch.no_grad():
            image_embed = self.base_model.get_image_embedding(image)  # [1, 512]
        image_embed = image_embed.detach()

        # Baseline: zero embeddings
        baseline = torch.zeros_like(token_embeds)

        # Integrated Gradients
        ig = IntegratedGradients(
            lambda embeds: self._forward_text_only(embeds, attention_mask, image_embed)
        )
        attributions = ig.attribute(
            token_embeds,
            baselines=baseline,
            target=target_class,
            n_steps=n_steps,
        )

        # Sum attributions across embedding dimension
        attr_scores = attributions.sum(dim=-1).squeeze(0).cpu().detach().numpy()  # [seq_len]

        # Decode tokens
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())

        # Pair tokens with scores (skip padding)
        result = []
        for tok, score, mask_val in zip(tokens, attr_scores, attention_mask[0].cpu().tolist()):
            if mask_val == 1 and tok not in ["[PAD]", "[CLS]", "[SEP]"]:
                result.append((tok, float(score)))

        return result


def visualize_text_attribution(
    attributions: List[Tuple[str, float]],
    disease_name: str,
    top_k: int = 20,
):
    """Visualize top-k important words as a horizontal bar chart."""
    # Sort by absolute attribution
    sorted_attrs = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:top_k]
    sorted_attrs = sorted_attrs[::-1]  # Reverse for plotting

    tokens = [a[0] for a in sorted_attrs]
    scores = [a[1] for a in sorted_attrs]
    colors = ["green" if s > 0 else "red" for s in scores]

    fig, ax = plt.subplots(figsize=(10, max(4, len(tokens) * 0.4)))
    ax.barh(range(len(tokens)), scores, color=colors, alpha=0.7)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=10)
    ax.set_xlabel("Attribution Score")
    ax.set_title(f"Text Attribution — {disease_name}")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.show()


print("✅ TextAttributor class defined.")

In [ ]:
# ============================================================
# 12.4 Text Attribution Examples
# ============================================================

text_attributor = TextAttributor(model, tokenizer, DEVICE)

# Generate examples for a few validation samples
for sample_idx in range(min(3, len(val_dataset))):
    sample = val_dataset[sample_idx]
    image = sample["image"].unsqueeze(0).to(DEVICE)
    labels = sample["labels"].numpy()
    text = df_val.iloc[sample_idx]["text"]

    # Get prediction
    with torch.no_grad():
        input_ids = sample["input_ids"].unsqueeze(0).to(DEVICE)
        attention_mask = sample["attention_mask"].unsqueeze(0).to(DEVICE)
        outputs = model(image, input_ids, attention_mask)
        probs_sample = torch.sigmoid(outputs["logits"]).cpu().numpy()[0]

    # Choose target class
    positive_classes = np.where(labels > 0.5)[0]
    if len(positive_classes) > 0:
        target_class = int(positive_classes[0])
    else:
        target_class = int(np.argmax(probs_sample))

    print(f"\n{'='*60}")
    print(f"📝 Sample {sample_idx + 1} — Target: {DISEASE_LABELS[target_class]} (prob={probs_sample[target_class]:.3f})")
    print(f"   Report: {text[:200]}...")
    print(f"{'='*60}")

    try:
        attributions = text_attributor.attribute(
            text=text,
            image=image,
            target_class=target_class,
            n_steps=30,
        )
        visualize_text_attribution(
            attributions,
            disease_name=DISEASE_LABELS[target_class],
            top_k=15,
        )
    except Exception as e:
        print(f"   ⚠️ Attribution failed: {e}")

---
## 13. Single Sample Inference

In [ ]:
# ============================================================
# 13.1 Predict Single Sample
# ============================================================

def predict_single_sample(
    model: nn.Module,
    image_path: str,
    report_text: str,
    tokenizer,
    transforms: T.Compose,
    thresholds: np.ndarray,
    disease_names: List[str],
    device: torch.device,
    generate_gradcam: bool = True,
    generate_text_attr: bool = True,
) -> Dict[str, Any]:
    """Complete inference pipeline for a single sample.
    
    Args:
        model: trained ImageTextClassifier
        image_path: path to chest X-ray image
        report_text: radiology report text
        tokenizer: BioClinicalBERT tokenizer
        transforms: image preprocessing transforms
        thresholds: per-class thresholds
        disease_names: list of disease names
        device: compute device
        generate_gradcam: whether to generate GradCAM
        generate_text_attr: whether to generate text attribution
    
    Returns:
        dict with probabilities, predictions, GradCAM, text attribution
    """
    model.eval()

    # ── Preprocess image ───────────────────────────────────────
    image = Image.open(image_path).convert("RGB")
    image_tensor = transforms(image).unsqueeze(0).to(device)

    # ── Preprocess text ────────────────────────────────────────
    cleaned_text = clean_report(report_text)
    encoding = tokenizer(
        cleaned_text,
        padding="max_length",
        truncation=True,
        max_length=CONFIG["max_text_len"],
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # ── Forward pass ───────────────────────────────────────────
    with torch.no_grad():
        outputs = model(image_tensor, input_ids, attention_mask)
        probs = torch.sigmoid(outputs["logits"]).cpu().numpy()[0]

    # ── Thresholded predictions ────────────────────────────────
    preds = (probs >= thresholds).astype(int)

    # ── "No Finding" conflict suppression ─────────────────────
    # If any disease is predicted positive, suppress "No Finding"
    no_finding_idx = disease_names.index("No Finding") if "No Finding" in disease_names else 0
    if preds.sum() > 1 and preds[no_finding_idx] == 1:
        # Check if any disease other than "No Finding" is positive
        other_positive = preds.copy()
        other_positive[no_finding_idx] = 0
        if other_positive.sum() > 0:
            preds[no_finding_idx] = 0
            print("   ⚡ 'No Finding' suppressed (conflict with positive disease predictions)")

    result = {
        "probabilities": {name: float(p) for name, p in zip(disease_names, probs)},
        "predictions": {name: int(p) for name, p in zip(disease_names, preds)},
        "thresholds": {name: float(t) for name, t in zip(disease_names, thresholds)},
    }

    # ── GradCAM ────────────────────────────────────────────────
    if generate_gradcam:
        try:
            target_class = int(np.argmax(probs))
            gradcam = ViTGradCAM(model, target_block_idx=-1)
            heatmap = gradcam.generate(image_tensor, input_ids, attention_mask, target_class)
            result["gradcam"] = heatmap
            result["gradcam_target"] = disease_names[target_class]
        except Exception as e:
            result["gradcam"] = None
            result["gradcam_error"] = str(e)

    # ── Text Attribution ───────────────────────────────────────
    if generate_text_attr:
        try:
            target_class = int(np.argmax(probs))
            attr = TextAttributor(model, tokenizer, device)
            attributions = attr.attribute(
                text=cleaned_text,
                image=image_tensor,
                target_class=target_class,
                n_steps=30,
            )
            # Top 10 important words
            sorted_attrs = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:10]
            result["important_words"] = sorted_attrs
            result["text_attr_target"] = disease_names[target_class]
        except Exception as e:
            result["important_words"] = None
            result["text_attr_error"] = str(e)

    return result


print("✅ predict_single_sample function defined.")

In [ ]:
# ============================================================
# 13.2 Single Sample Inference Demo
# ============================================================

# Use first validation sample as demo
demo_idx = 0
demo_row = df_val.iloc[demo_idx]
demo_image_path = demo_row["image_path"]
demo_text = demo_row["text"]

print(f"🔍 Demo inference on sample {demo_idx}")
print(f"   Image: {demo_image_path}")
print(f"   Report: {demo_text[:150]}...")

demo_result = predict_single_sample(
    model=model,
    image_path=demo_image_path,
    report_text=demo_text,
    tokenizer=tokenizer,
    transforms=val_transforms,
    thresholds=ema_thresholds,
    disease_names=DISEASE_LABELS,
    device=DEVICE,
    generate_gradcam=True,
    generate_text_attr=True,
)

print(f"\n📊 Predictions:")
for name in DISEASE_LABELS:
    prob = demo_result["probabilities"][name]
    pred = demo_result["predictions"][name]
    thresh = demo_result["thresholds"][name]
    status = "✅" if pred == 1 else "❌"
    print(f"   {status} {name:<20s}: prob={prob:.4f} (thresh={thresh:.3f})")

if demo_result.get("important_words"):
    print(f"\n📝 Top important words for {demo_result['text_attr_target']}:")
    for token, score in demo_result["important_words"]:
        print(f"   {token:<20s}: {score:+.4f}")

# Visualize GradCAM
if demo_result.get("gradcam") is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    orig_img = Image.open(demo_image_path).convert("RGB").resize((224, 224))
    axes[0].imshow(orig_img)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(np.array(orig_img) / 255.0)
    axes[1].imshow(demo_result["gradcam"], cmap="jet", alpha=0.5)
    axes[1].set_title(f"GradCAM — {demo_result['gradcam_target']}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

---
## 14. Feature Extraction

In [ ]:
# ============================================================
# 14.1 Extract Embeddings
# ============================================================

@torch.no_grad()
def extract_embeddings(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Dict[str, np.ndarray]:
    """Extract image, text, and fused embeddings from the model.
    
    Returns:
        dict with keys:
            - image_embeddings: [N, 512]
            - text_embeddings: [N, 768]
            - fused_embeddings: [N, 1280]
            - study_ids: [N]
    """
    model.eval()
    all_image_embeds = []
    all_text_embeds = []
    all_fused_embeds = []
    all_study_ids = []

    for batch in tqdm(loader, desc="Extracting embeddings"):
        images = batch["image"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)

        with autocast(enabled=CONFIG["mixed_precision"]):
            outputs = model(images, input_ids, attention_mask)

        all_image_embeds.append(outputs["image_embed"].cpu().numpy())
        all_text_embeds.append(outputs["text_embed"].cpu().numpy())
        all_fused_embeds.append(outputs["fused_embed"].cpu().numpy())
        all_study_ids.extend(batch["study_id"].tolist())

    return {
        "image_embeddings": np.concatenate(all_image_embeds, axis=0),
        "text_embeddings": np.concatenate(all_text_embeds, axis=0),
        "fused_embeddings": np.concatenate(all_fused_embeds, axis=0),
        "study_ids": np.array(all_study_ids),
    }


print("🔄 Extracting validation set embeddings...")
embeddings = extract_embeddings(model, val_loader, DEVICE)

print(f"\n📊 Embedding shapes:")
print(f"   Image:  {embeddings['image_embeddings'].shape}")
print(f"   Text:   {embeddings['text_embeddings'].shape}")
print(f"   Fused:  {embeddings['fused_embeddings'].shape}")
print(f"   IDs:    {embeddings['study_ids'].shape}")

# Save embeddings
np.save(
    os.path.join(CONFIG["output_dir"], "multimodal_embeddings.npy"),
    embeddings["fused_embeddings"],
)

# Save study IDs
pd.DataFrame({"study_id": embeddings["study_ids"]}).to_csv(
    os.path.join(CONFIG["output_dir"], "study_ids.csv"),
    index=False,
)

print("\n✅ Embeddings and study IDs saved.")

---
## 15. Save All Artifacts

In [ ]:
# ============================================================
# 15.1 Save Final Metrics
# ============================================================

# Choose best metrics (typically EMA + TTA)
final_metrics = {
    "baseline": baseline_metrics,
    "ema": ema_metrics,
    "ema_tta": tta_metrics,
}

# Convert numpy types for JSON serialization
def convert_numpy(obj):
    """Recursively convert numpy types to native Python types."""
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


with open(os.path.join(CONFIG["output_dir"], "final_metrics.json"), "w") as f:
    json.dump(convert_numpy(final_metrics), f, indent=2)

print("✅ Final metrics saved.")

In [ ]:
# ============================================================
# 15.2 Save Configuration
# ============================================================

with open(os.path.join(CONFIG["output_dir"], "config.json"), "w") as f:
    json.dump(convert_numpy(CONFIG), f, indent=2)

print("✅ Configuration saved.")

In [ ]:
# ============================================================
# 15.3 Artifact Summary
# ============================================================

print(f"\n{'='*70}")
print(f"📦 SAVED ARTIFACTS")
print(f"{'='*70}")

artifact_dir = CONFIG["output_dir"]
expected_files = [
    "best_model.pth",
    "best_model_ema.pth",
    "checkpoint_last.pth",
    "thresholds.json",
    "training_history.pkl",
    "final_metrics.json",
    "config.json",
    "multimodal_embeddings.npy",
    "study_ids.csv",
    "training_history.png",
    "roc_curves.png",
    "confusion_matrices.png",
    "auroc_bar.png",
    "threshold_curves.png",
]

for filename in expected_files:
    filepath = os.path.join(artifact_dir, filename)
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"   ✅ {filename:<35s} ({size_mb:.2f} MB)")
    else:
        print(f"   ❌ {filename:<35s} (MISSING)")

# Check gradcam directory
gradcam_dir = os.path.join(artifact_dir, "gradcam_examples")
if os.path.exists(gradcam_dir):
    gradcam_files = os.listdir(gradcam_dir)
    print(f"   ✅ gradcam_examples/              ({len(gradcam_files)} files)")

print(f"\n   📁 All artifacts saved to: {artifact_dir}")

---
## 16. Final Summary

This notebook implements a complete multimodal medical AI pipeline for chest X-ray disease classification.

In [ ]:
# ============================================================
# 16.1 Final Report
# ============================================================

print(f"""
{'='*70}
🏥 MULTIMODAL MEDICAL FOUNDATION MODEL — FINAL REPORT
{'='*70}

📌 Architecture:
   Image Encoder:  BiomedCLIP ViT-B/16 (512-d)
   Text Encoder:   BioClinicalBERT (768-d, CLS token)
   Fusion:         Concatenation → MLP (1280 → 512 → 256 → 6)

📌 Training:
   Epochs trained:  {len(history['train_loss'])}
   Best Val AUROC:  {best_auroc:.4f}
   Total time:      {sum(history['epoch_time']):.0f}s

📌 Best Results (EMA + TTA):
   Macro AUROC:        {tta_metrics['aggregate']['macro_auroc']:.4f}
   Macro F1:           {tta_metrics['aggregate']['macro_f1']:.4f}
   Exact Match Acc:    {tta_metrics['aggregate']['exact_match_accuracy']:.4f}
   Hamming Acc:        {tta_metrics['aggregate']['hamming_accuracy']:.4f}

📌 Per-Disease AUROC:
""")

for name in DISEASE_LABELS:
    auroc_val = tta_metrics["per_disease"][name]["auroc"]
    print(f"   {name:<20s}: {auroc_val:.4f}")

print(f"""
📌 Artifacts saved to: {CONFIG['output_dir']}

{'='*70}
✅ PIPELINE COMPLETE
{'='*70}
""")

In [ ]:
# ============================================================
# Cleanup
# ============================================================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("🧹 Memory cleaned up.")